# The ML challenge: you choose the technique

You now know ten techniques. This session you decide which one to use.

There are four datasets below. Each comes with questions. For every question:

1. **Decide what kind of question it is** (use the guide below).
2. **Pick a technique and say why** in one sentence.
3. **Write the code** and get an answer.
4. **Try a second technique.** Do both give the same answer?

Work in your breakout group. Pick **at least two** datasets. At the end, each
group reports back: which technique, what score, and did the techniques agree?

## Which kind of question is it?

```
Is there an answer column to predict?
├─ No  → find groups:   K-means, DBSCAN
└─ Yes → is the answer a number or a category?
         ├─ Number   → LinearRegression, DecisionTreeRegressor, KNeighborsRegressor
         └─ Category → LogisticRegression, KNeighborsClassifier, SVC,
                       GaussianNB, DecisionTreeClassifier

Too many columns to look at?   → PCA first, to see the data in 2-D
Need to explain WHY?           → a decision tree (rules) or
                                 logistic regression (coefficients)
```

## The five-line recipe

Every scikit-learn model works the same way. Learn this once:

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1)

model = DecisionTreeClassifier()        # the only line you change
model.fit(X_train, y_train)             # learn from the training rows
model.score(X_test, y_test)             # test on rows it has never seen
```

`score` gives **accuracy** for a category and **R²** for a number.

Two reminders:

- K-NN, SVC and K-means measure distances, so a column measured in
  thousands drowns out one measured in decimals. Scale first with
  `StandardScaler`. Trees do not need it.
- Fit the scaler on the training rows only, then use it on both.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text

# Where the CSV files are: the shared data/ folder at the top of the repo.
# Change this if yours are somewhere else.
DATA_FOLDER = "../../../data/"

# Dataset 1: handwritten digits

1,797 small pictures of handwritten digits, 0 to 9. Each picture is 8 × 8
pixels, so each row has **64 columns**, one per pixel, from 0 (white) to 16
(black). It ships with scikit-learn, so there is nothing to download.

Source: UCI Optical Recognition of Handwritten Digits, loaded with
`load_digits()`.

In [ ]:
digits = load_digits()
X = digits.data        # 64 pixel columns
y = digits.target      # the digit it really is

print("Rows and columns:", X.shape)

# Show the first ten pictures
fig, axes = plt.subplots(1, 10, figsize=(10, 1.5))
for i in range(10):
    axes[i].imshow(digits.images[i], cmap="gray_r")
    axes[i].set_title(y[i])
    axes[i].axis("off")
plt.show()

In [ ]:
print(digits.keys())
print("images:", digits.images.shape)
print("data:  ", digits.data.shape)
print("target:", digits.target.shape)

In [ ]:
print(digits.images[0].astype(int))

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(digits.images[0], cmap="gray_r")
plt.title("This digit is: " + str(digits.target[0]))
plt.show()

### Q1. 64 columns is too many to look at. Can you see the digits in a 2-D picture? Do they form groups?

Which technique turns 64 columns into 2?

**Kind of question:** ...
**Technique I chose, and why:** ...
**Second technique:** ...
**Answer, and did they agree?** ...

### Q2. How accurately can a computer read a handwritten digit? Which technique does best?

Is this a number or a category? Try at least two techniques.

**Kind of question:** ...
**Technique I chose, and why:** ...
**Second technique:** ...
**Answer, and did they agree?** ...

### Q3 (stretch). Hide the answers. Does a clustering method find the ten digits on its own?

Use only `X`, never `y`, to make the groups. Then compare your groups with `y`.

**Kind of question:** ...
**Technique I chose, and why:** ...
**Second technique:** ...
**Answer, and did they agree?** ...

# Dataset 2: breast cancer

569 tumour samples. Each has **30 measurements** taken from a scan of the
cells (radius, texture, area, smoothness, and so on). Each is labelled
malignant (cancer) or benign (not cancer). It ships with scikit-learn.

Source: Breast Cancer Wisconsin (Diagnostic), loaded with
`load_breast_cancer()`.

**Watch out:** in this dataset `0` means **malignant** and `1` means
**benign**. That is the opposite of what most people guess.

In [ ]:
cancer = load_breast_cancer(as_frame=True)
X = cancer.data        # 30 measurement columns
y = cancer.target      # 0 = malignant, 1 = benign

print("Rows and columns:", X.shape)
print()
print("How many of each:")
print(y.value_counts().rename({0: "malignant", 1: "benign"}))

### Q1. Can you tell malignant from benign? How accurate can you get?

Try at least three techniques. Do they agree?

**Kind of question:** ...
**Technique I chose, and why:** ...
**Second technique:** ...
**Answer, and did they agree?** ...

### Q2. A model can be wrong in two ways. Which mistake is worse here? Which model makes fewer of them?

- It says **benign** but the tumour is **malignant**: a missed cancer.
- It says **malignant** but the tumour is **benign**: a false alarm.

Count the missed cancers for each model. Is the most accurate model also the
one that misses fewest?

**Kind of question:** ...
**Technique I chose, and why:** ...
**Second technique:** ...
**Answer, and did they agree?** ...

### Q3 (stretch). Run PCA with 2 components on the raw data, then on scaled data. What changes, and why?

Hint: look at `X.var()`.

**Kind of question:** ...
**Technique I chose, and why:** ...
**Second technique:** ...
**Answer, and did they agree?** ...

# Dataset 3: the Titanic

891 passengers on the Titanic, with class of ticket, sex, age, family on
board, fare paid, and whether they survived.

Source: the Kaggle "Titanic" training file, `titanic_train.csv`.

The data needs a little cleaning before a model will accept it:

- `Sex` is text. Models need numbers, so make `is_female`: 1 or 0.
- `Age` is missing for some passengers. Fill the gaps with the middle
  (median) age.

In [ ]:
titanic = pd.read_csv(DATA_FOLDER + "titanic_train.csv")

# Turn "female" / "male" into 1 / 0
titanic["is_female"] = (titanic["Sex"] == "female").astype(int)

# Fill missing ages with the median age
median_age = titanic["Age"].median()
titanic["Age"] = titanic["Age"].fillna(median_age)

feature_columns = ["Pclass", "is_female", "Age", "SibSp", "Parch", "Fare"]
X = titanic[feature_columns]
y = titanic["Survived"]   # 1 = survived, 0 = died

print("Rows and columns:", X.shape)
print("Missing values left:", X.isna().sum().sum())
titanic.head()

### Q1. Can you predict who survived? Beat the lazy guess.

The lazy guess is "everybody died". Work out how often that guess is right
first. Your model must do better than that to be worth anything.

**Kind of question:** ...
**Technique I chose, and why:** ...
**Second technique:** ...
**Answer, and did they agree?** ...

### Q2. *Why* did people survive? Do two different techniques give the same reasons?

Pick two techniques that can explain themselves. Read what each one learned.

**Kind of question:** ...
**Technique I chose, and why:** ...
**Second technique:** ...
**Answer, and did they agree?** ...

### Q3 (stretch). Ignore `Survived`. Are there natural groups of passengers? Did the groups survive at different rates?

**Kind of question:** ...
**Technique I chose, and why:** ...
**Second technique:** ...
**Answer, and did they agree?** ...

# Dataset 4: bike share, one row per day

731 days (2011 and 2012) of a bike-share scheme in Washington DC: the
season, the weather, whether it was a working day, and how many bikes were
rented (`cnt`).

Source: UCI Bike Sharing Dataset, `day.csv` (Fanaee-T and Gama, 2013),
saved here as `bikeshare-day.csv`.

Columns you need to know:

| Column | Meaning |
|---|---|
| `yr` | 0 = 2011, 1 = 2012 |
| `weathersit` | 1 = clear, 2 = mist, 3 = light rain or snow |
| `temp`, `hum`, `windspeed` | scaled to between 0 and 1 |
| `casual`, `registered` | rentals by casual users and by members |
| `cnt` | total rentals that day: the answer |

In [ ]:
bikes = pd.read_csv(DATA_FOLDER + "bikeshare-day.csv")

feature_columns = ["season", "yr", "mnth", "holiday", "weekday", "workingday",
                   "weathersit", "temp", "hum", "windspeed"]
X = bikes[feature_columns]
y = bikes["cnt"]

print("Rows and columns:", X.shape)
bikes.head()

### Q1. How many bikes will be rented on a given day? Compare three techniques.

Is this a number or a category? `score` now gives R², where 1 is perfect.
Also work out how many bikes each model is off by, on average.

**Kind of question:** ...
**Technique I chose, and why:** ...
**Second technique:** ...
**Answer, and did they agree?** ...

### Q2. Which features matter most for the number of rentals?

**Kind of question:** ...
**Technique I chose, and why:** ...
**Second technique:** ...
**Answer, and did they agree?** ...

### Q3 (stretch). A classmate adds `casual` and `registered` to the features and gets an R² of almost 1. Is their model brilliant?

Try it. Then explain what went wrong.

**Kind of question:** ...
**Technique I chose, and why:** ...
**Second technique:** ...
**Answer, and did they agree?** ...

# Report back

Fill in one row per question your group answered.

| Dataset | Question | Technique 1 and score | Technique 2 and score | Did they agree? |
|---|---|---|---|---|
| | | | | |
| | | | | |
| | | | | |